# Credit Scorecard Development
## Notebook 4 — Model Validation

Model validation covers three pillars:

| Metric | Description | Benchmark |
|--------|-------------|----------|
| **AUC** | Area Under ROC Curve | > 0.70 |
| **Gini** | Gini = 2×AUC − 1 | > 0.40 |
| **KS**  | Max separation between default/non-default CDFs | > 0.30 |
| **PSI** | Population Stability Index (monitoring) | < 0.10 stable |

**Validation strategy:** Out-of-time (OOT) split — the last 30% of data by time serves as test set,
mimicking real deployment conditions.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import (generate_credit_data, train_test_split_time, WoEBinner,
                 CreditScorecard, validation_report, plot_roc_ks, population_stability_index)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

df = generate_credit_data(n_samples=15_000, random_state=42)
df_train, df_test = train_test_split_time(df, test_size=0.3)

features = ['age', 'annual_income', 'employment_years', 'loan_amount',
            'loan_term', 'credit_utilization', 'num_delinquencies',
            'num_credit_accounts', 'debt_to_income']

binner = WoEBinner(max_bins=10).fit(df_train[features], df_train['default'])
iv_summary = binner.iv_summary()
selected = iv_summary[iv_summary['iv'] >= 0.02]['feature'].tolist()
woe_feats = [f'{f}_woe' for f in selected]

X_train = binner.transform(df_train[features])[woe_feats]
X_test  = binner.transform(df_test[features])[woe_feats]
y_train, y_test = df_train['default'], df_test['default']

scorecard = CreditScorecard(base_score=600, base_odds=50, pdo=20).fit(X_train, y_train)

pd_train = scorecard.predict_proba(X_train)[:, 1]
pd_test  = scorecard.predict_proba(X_test)[:, 1]

print('Model trained successfully.')

## 1. Discrimination Metrics — Train vs Test

In [ ]:
report_train = validation_report(y_train.values, pd_train, label='Train (In-Time)')
report_test  = validation_report(y_test.values,  pd_test,  label='Test  (Out-of-Time)')

report = pd.concat([report_train, report_test]).reset_index(drop=True)
print('=== Model Validation Report ===')
print(report.to_string(index=False))

# Overfitting check
gini_gap = abs(report_train['Gini'].values[0] - report_test['Gini'].values[0])
print(f'\nGini gap (Train-Test): {gini_gap:.4f}  ← should be < 0.05 for stable model')

## 2. ROC Curve & KS Plot

In [ ]:
fig = plot_roc_ks(y_train.values, pd_train, title='Train (In-Time)')
plt.savefig('../plots/04_roc_ks_train.png', bbox_inches='tight')
plt.show()

fig = plot_roc_ks(y_test.values, pd_test, title='Test (Out-of-Time)')
plt.savefig('../plots/04_roc_ks_test.png', bbox_inches='tight')
plt.show()

## 3. Population Stability Index (PSI)

In [ ]:
score_train = scorecard.predict_score(X_train)
score_test  = scorecard.predict_score(X_test)

psi_score = population_stability_index(score_train, score_test, bins=10)
psi_label = 'Stable' if psi_score < 0.10 else ('Monitor' if psi_score < 0.25 else 'Unstable')

print(f'PSI (Score): {psi_score:.4f} → {psi_label}')

# Visual PSI
breakpoints = np.percentile(score_train, np.linspace(0, 100, 11))
breakpoints[0], breakpoints[-1] = -np.inf, np.inf

train_pct = np.histogram(score_train, bins=breakpoints)[0] / len(score_train) * 100
test_pct  = np.histogram(score_test,  bins=breakpoints)[0] / len(score_test)  * 100

fig, ax = plt.subplots(figsize=(12, 5))
x = range(10)
width = 0.4
ax.bar([i - width/2 for i in x], train_pct, width, label='Train (Base)', color='#2980b9', alpha=0.8)
ax.bar([i + width/2 for i in x], test_pct,  width, label='Test  (Current)', color='#e74c3c', alpha=0.8)
ax.set_xlabel('Score Decile')
ax.set_ylabel('% of Population')
ax.set_title(f'Population Stability Index — PSI = {psi_score:.4f} ({psi_label})', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/04_psi.png', bbox_inches='tight')
plt.show()

## 4. Score Decile Analysis (Gains Table)

In [ ]:
df_val = pd.DataFrame({'score': score_test, 'default': y_test.values})
df_val['decile'] = pd.qcut(df_val['score'], q=10, labels=False, duplicates='drop')
df_val['decile'] = 10 - df_val['decile']  # flip: decile 1 = highest risk

gains = (
    df_val.groupby('decile')
    .agg(n_loans=('default', 'count'),
         n_defaults=('default', 'sum'),
         min_score=('score', 'min'),
         max_score=('score', 'max'))
    .assign(
        default_rate=lambda x: (x['n_defaults'] / x['n_loans'] * 100).round(2),
        cum_default_pct=lambda x: (x['n_defaults'].cumsum() / x['n_defaults'].sum() * 100).round(2),
    )
)

print('=== Score Decile Analysis (Test Set) ===')
print(gains.to_string())